In [14]:
import sys
sys.path.append('../')

In [15]:
import uuid
from dotenv import load_dotenv
import os
from src.utils.excel import parse_excel_file

# Load environment variables
load_dotenv()
print("✓ Environment variables loaded")

✓ Environment variables loaded


In [16]:
# Model configuration
client_type = 'openai'  # or antropic, groq
model_name = 'gpt-4o-mini'

print(f"\nMODEL CONFIGURATION")
print(f"Client Type: {client_type}")
print(f"Model Name:  {model_name}")


MODEL CONFIGURATION
Client Type: openai
Model Name:  gpt-4o-mini


In [17]:
# Run configuration
run_id = str(uuid.uuid4())
SAVE_EVERY = 3
MAX_WORKERS = 2
base_output_dir = '../data/evaluation_output'
base_filename = f'eval_{client_type}_{model_name}_{run_id}'
results_file = os.path.join(base_output_dir, f"{base_filename}.json")
adjusted_file = os.path.join(base_output_dir, f"{base_filename}_adjusted.json")

# Create output directory
os.makedirs(base_output_dir, exist_ok=True)

print(f"\nRUN CONFIGURATION")
print(f"Run ID:       {run_id}")
print(f"Save Every:   {SAVE_EVERY} records")
print(f"Max Workers:  {MAX_WORKERS}")
print(f"Output Dir:   {base_output_dir}")
print(f"Results File: {os.path.basename(results_file)}")
print(f"Adjusted File: {os.path.basename(adjusted_file)}")


RUN CONFIGURATION
Run ID:       c63b7305-5c3e-4eb9-96b0-0226b4861bd0
Save Every:   3 records
Max Workers:  2
Output Dir:   ../data/evaluation_output
Results File: eval_openai_gpt-4o-mini_c63b7305-5c3e-4eb9-96b0-0226b4861bd0.json
Adjusted File: eval_openai_gpt-4o-mini_c63b7305-5c3e-4eb9-96b0-0226b4861bd0_adjusted.json


In [18]:
# Load and configure evaluation data
dataset_path = "../data/Dset_Eval_KW_Alignment_Eval_def.xlsx"
total_records = parse_excel_file(dataset_path)
records = total_records[:3]
print(f"\nEVAL DATASET")
print(f"✓ Dataset loaded successfully")
print(f"Total Records Available: {len(total_records):,}")
print(f"Records Selected for Evaluation: {len(records):,}")


EVAL DATASET
✓ Dataset loaded successfully
Total Records Available: 202
Records Selected for Evaluation: 3


In [19]:
from src.pipelines import EntityExtractionPipeline, DirectWikidataLinkingPipeline

pipeline = EntityExtractionPipeline(client_type, model_name=model_name)

# pipeline = DirectWikidataLinkingPipeline(model_name=model_name)

In [20]:
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

file_lock = threading.Lock()

def process_keyword(record_data):
    record_idx, kw_idx, language, title_or, abstract_or, kw_label = record_data
    
    try:
        llm_uris = pipeline.run(
            language=language,
            title=title_or,
            abstract=abstract_or,
            keywords=kw_label
        )
        return record_idx, kw_idx, llm_uris, None
    except Exception as e:
        print(f"LLM URIs cannot be computed for record {record_idx}, keyword {kw_idx}: {e}")
        return record_idx, kw_idx, [], str(e)

tasks_data = []
for record_idx, record in enumerate(records):
    for kw_idx, kw in enumerate(record['kws']):
        task_data = (
            record_idx,
            kw_idx,
            record['language'],
            record['title_eng'],
            record['abstract_eng'],
            kw['label']
        )
        tasks_data.append(task_data)

print(f"Preparati {len(tasks_data)} task da processare")

Preparati 17 task da processare


In [21]:

# Processa in parallelo
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Sottometti tutti i task con dati già copiati
    future_to_task = {
        executor.submit(process_keyword, task_data): task_data[0:2]  # record_idx, kw_idx
        for task_data in tasks_data
    }
    
    processed_count = 0
    with tqdm(total=len(tasks_data), desc="Processing keywords") as pbar:
        for future in as_completed(future_to_task):
            record_idx, kw_idx, llm_uris, error = future.result()
            
            # Aggiorna i risultati
            records[record_idx]['kws'][kw_idx]['llm_uris'] = llm_uris
            
            processed_count += 1
            pbar.update(1)
            
            # Salvataggio periodico thread-safe
            if processed_count % SAVE_EVERY == 0:
                with file_lock:
                    # print(f"Saving partial results at {processed_count} processed keywords")
                    with open(results_file, 'w', encoding='utf-8') as f:
                        json.dump(records, f, ensure_ascii=False, indent=2)

# Salvataggio finale
print("All keywords processed, saving final results.")
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

Processing keywords: 100%|██████████| 17/17 [00:32<00:00,  1.91s/it]

All keywords processed, saving final results.


In [22]:
new_records = []

with open(results_file, 'r', encoding='utf-8') as f:
    records = json.load(f)

for record in records:
    new_record = record
    for i, kw in enumerate(record['kws']):
        if len(kw['llm_uris']) == 3 and kw['llm_uris'][1] == "wikidata":
            new_record['kws'][i]['llm_uris'] = ['.'.join(kw['llm_uris'])]
    new_records.append(new_record)

with open(adjusted_file, 'w', encoding='utf-8') as f:
    json.dump(new_records, f, ensure_ascii=False, indent=2)